# f6_m00_preparacion.ipynb

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 6 — Interpretabilidad y Evaluación Final |
| **Módulo** | M00 — Preparación |

---

## 🎯 Qué hace

Construye `meta_test.parquet` — tabla de metadatos del conjunto de test para uso
en todos los módulos de Fase 6. Cruza los índices de test con `df_eda_final` para
recuperar `titulacion`, `rama`, `per_id_ficticio` y otras variables de contexto
que no están en `X_test_prep`. Debe ejecutarse **antes** de cualquier otro módulo de Fase 6.



📌 **Genera también** `metricas_modelo.json` con métricas del modelo + cifras canónicas del test (`n_test=6.596`, filtrado 2010-2020). Este JSON es la **única fuente de verdad** para los datos mostrados en la app Streamlit (Fase 7) — sin valores hardcodeados.

## 📋 Requisitos

- `data/04_eda/df_eda_final.parquet` — dataset EDA con titulación y contexto
- `data/05_modelado/X_test_prep.parquet` — índices del conjunto de test (6.725 filas)
- `data/03_features/df_exp_automl_target.parquet` — con `per_id_ficticio`
- `data/03_features/df_alumno_limpio.parquet` — datos a nivel alumno
- `data/05_modelado/y_test.parquet` — etiquetas reales del test
- Entorno: `tfm_abandono` (pandas, numpy)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/06_evaluacion/meta_test.parquet` | Metadatos del test: titulación, rama, per_id_ficticio, abandono, flags |

## 🔄 Flujo

```
data/04_eda/df_eda_final.parquet          ┐
data/05_modelado/X_test_prep.parquet      ├─→ merge por índice → meta_test.parquet
data/03_features/df_exp_automl_target     ┘
    ↓ Verificación de ficheros necesarios
    ↓ Extracción de metadatos del test
    ↓ Agregación a nivel alumno
    ↓ Flag de cautela (titulaciones con sesgo temporal)
    ↓ Verificación final de shape e índices
    → data/06_evaluacion/meta_test.parquet
```

## ➡️ Siguiente

`f6_m01a_shap_global.ipynb` — cálculo de valores SHAP sobre el conjunto de test


In [1]:
# ============================================================================
# CELDA 1: IMPORTS Y RUTAS
# ============================================================================
# Detección robusta de ROOT subiendo niveles hasta encontrar src/
# Verifica existencia de los 4 ficheros necesarios antes de continuar
# ============================================================================

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# ROOT detection estándar TFM
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config_entorno import RAMAS_NOMBRES, NOMBRES_LEGIBLES_FEATURES

# Rutas de datos
RUTA_EDA   = ROOT / 'data' / '04_eda'
RUTA_FEAT  = ROOT / 'data' / '03_features'
RUTA_MODEL = ROOT / 'data' / '05_modelado'

RUTA_EVAL = ROOT / 'data' / '06_evaluacion'
RUTA_EVAL.mkdir(exist_ok=True)  # crea la carpeta si no existe

# Verificar que todos los ficheros necesarios existen antes de continuar
ficheros_necesarios = {
    'df_eda_final':         RUTA_EDA   / 'df_eda_final.parquet',
    'X_test_prep':          RUTA_MODEL / 'X_test_prep.parquet',
    'df_exp_automl_target': RUTA_FEAT  / 'df_exp_automl_target.parquet',
    'df_alumno_limpio':     RUTA_FEAT  / 'df_alumno_limpio.parquet',
}

print('Verificando ficheros necesarios:')
todos_ok = True
for nombre, ruta in ficheros_necesarios.items():
    ok = ruta.exists()
    print(f'  {"✅" if ok else "❌"}  {nombre}')
    if not ok:
        todos_ok = False

assert todos_ok, 'Faltan ficheros necesarios. Revisar rutas.'
print(f'\n✅ Todos los ficheros encontrados')
print(f'ROOT: {ROOT}')

Verificando ficheros necesarios:
  ✅  df_eda_final
  ✅  X_test_prep
  ✅  df_exp_automl_target
  ✅  df_alumno_limpio

✅ Todos los ficheros encontrados
ROOT: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI


In [2]:
# ============================================================================
# CELDA 2: CARGA DE DATOS
# ============================================================================
# df_eda_final: 33.621 filas, índice 0-33620, contiene titulacion
# df_exp:       33.621 filas, mismo índice posicional — aporta per_id_ficticio
# df_lim:       109.568 filas (nivel curso académico) — aporta curso_aca_ini,
#               vive_fuera, cupo. Se agrega a nivel alumno en Celda 4.
# X_test:       6.725 filas — sus índices son posiciones en df_eda_final
# ============================================================================

# Dataset EDA final — contiene titulacion, alineado con modelado
df_eda = pd.read_parquet(RUTA_EDA / 'df_eda_final.parquet')

# Índices del conjunto de test (posiciones en df_eda_final)
X_test = pd.read_parquet(RUTA_MODEL / 'X_test_prep.parquet')

# per_id_ficticio alineado con df_eda_final (índice posicional 0-33620)
df_exp = pd.read_parquet(RUTA_FEAT / 'df_exp_automl_target.parquet')[['per_id_ficticio']]

# Datos de alumno a nivel curso — se agrega por alumno en Celda 4
df_lim = pd.read_parquet(RUTA_FEAT / 'df_alumno_limpio.parquet')[
    ['per_id_ficticio', 'curso_aca_ini', 'vive_fuera', 'cupo']
]

print(f'df_eda_final:          {df_eda.shape}  — índice {df_eda.index.min()}-{df_eda.index.max()}')
print(f'X_test:                {X_test.shape}  — índice {X_test.index.min()}-{X_test.index.max()}')
print(f'df_exp (per_id):       {df_exp.shape}')
print(f'df_alumno_limpio:      {df_lim.shape}')

# Verificación crítica: df_exp y df_eda deben estar alineados
assert len(df_exp) == len(df_eda), \
    f'ERROR: df_exp ({len(df_exp)}) y df_eda ({len(df_eda)}) no coinciden'
print('\n✅ df_exp y df_eda alineados (mismo nº filas)')

df_eda_final:          (33621, 26)  — índice 0-33620
X_test:                (6725, 27)  — índice 16-33619
df_exp (per_id):       (33621, 1)
df_alumno_limpio:      (109568, 4)

✅ df_exp y df_eda alineados (mismo nº filas)


In [3]:
# ============================================================================
# CELDA 3: META BASE — extraer contexto de test desde df_eda_final
# ============================================================================
# X_test.index contiene posiciones (ej: 16823, 4904...) en df_eda_final.
# Con iloc extraemos exactamente esas filas y su contexto.
# Añadimos per_id_ficticio desde df_exp con el mismo mecanismo posicional.
# ============================================================================

# Columnas de contexto disponibles en df_eda_final
COLS_EDA = ['titulacion', 'rama', 'sexo', 'pais_nombre',
            'provincia', 'via_acceso', 'abandono']

# Extraer filas de test por posición
meta_base = df_eda.iloc[X_test.index][COLS_EDA].copy()
meta_base.index = X_test.index

# Añadir per_id_ficticio desde df_exp (mismo índice posicional)
meta_base['per_id_ficticio'] = df_exp.iloc[X_test.index]['per_id_ficticio'].values

print(f'meta_base shape:           {meta_base.shape}')
print(f'Titulaciones únicas:       {meta_base["titulacion"].nunique()}')
print(f'per_id_ficticio únicos:    {meta_base["per_id_ficticio"].nunique()}')
print(f'Tasa abandono en test:     {meta_base["abandono"].mean()*100:.1f}%')
print(f'\nPrimeras 3 filas:')
print(meta_base.head(3))

meta_base shape:           (6725, 8)
Titulaciones únicas:       40
per_id_ficticio únicos:    6603
Tasa abandono en test:     29.2%

Primeras 3 filas:
                             titulacion rama    sexo pais_nombre provincia  \
16823  Grado en Finanzas y Contabilidad   SO   Mujer      España  Castelló   
4904                Grado en Psicología   SA  Hombre      España  Castelló   
11906                 Grado en Medicina   SA   Mujer      España  Castelló   

                           via_acceso  abandono  per_id_ficticio  
16823  Pruebas acceso Bachiller Logse         0          1197964  
4904   Pruebas acceso Bachiller Logse         0           733138  
11906  Pruebas acceso Bachiller Logse         0           976958  


In [4]:
# ============================================================================
# CELDA 4: AGREGAR df_alumno_limpio A NIVEL ALUMNO
# ============================================================================
# df_alumno_limpio tiene 109.568 filas (una por alumno×año×titulación).
# Necesitamos una fila por per_id_ficticio para hacer el merge.
#
# Reglas de agregación:
#   curso_aca_ini → min: año de primera matrícula en cualquier titulación
#   vive_fuera    → moda: valor más frecuente durante sus años de estudio
#   cupo          → moda: tipo de cupo de acceso predominante
# ============================================================================

df_extra = (
    df_lim
    .groupby('per_id_ficticio')
    .agg(
        curso_aca_ini=('curso_aca_ini', 'min'),
        vive_fuera   =('vive_fuera',    lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        cupo         =('cupo',          lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    )
    .reset_index()
)

print(f'df_extra shape: {df_extra.shape}  (un alumno por fila)')
print(f'\nEjemplo primeras 3 filas:')
print(df_extra.head(3))
print(f'\ncupo — distribución:')
print(df_extra['cupo'].value_counts().head(8))

df_extra shape: (30872, 4)  (un alumno por fila)

Ejemplo primeras 3 filas:
   per_id_ficticio  curso_aca_ini  vive_fuera     cupo
0              106           2012        True  General
1              134           2009       False     None
2              426           2010       False  General

cupo — distribución:
cupo
General                   26323
Mayor 25 Años               594
Titulados                   287
Mayor 40años                 35
Mayor 45años                 31
Minusvalidos                 20
Deportistas Alto Nivel       18
Mayor 45 Años                18
Name: count, dtype: int64


In [5]:
# ============================================================================
# CELDA 5: MERGE Y CÁLCULO DE n_titulaciones
# ============================================================================
# Merge meta_base + df_extra por per_id_ficticio.
# n_titulaciones: cuántas carreras distintas cursó cada alumno.
# Se calcula sobre df_eda completo (no solo test) para ser representativo.
# ============================================================================

# Merge principal — añade curso_aca_ini, vive_fuera, cupo
meta_test = meta_base.merge(df_extra, on='per_id_ficticio', how='left')
meta_test.index = X_test.index  # restaurar índice tras merge

# Calcular n_titulaciones: nº de titulaciones distintas por alumno en df_eda
# df_exp tiene per_id_ficticio alineado con df_eda (mismo índice posicional)
df_eda_con_id = df_eda[['titulacion']].copy()
df_eda_con_id['per_id_ficticio'] = df_exp['per_id_ficticio'].values

n_tit_map = (
    df_eda_con_id
    .groupby('per_id_ficticio')['titulacion']
    .nunique()
    .rename('n_titulaciones')
    .reset_index()
)

# Añadir n_titulaciones y restaurar índice
meta_test = meta_test.merge(n_tit_map, on='per_id_ficticio', how='left')
meta_test.index = X_test.index

print(f'meta_test shape: {meta_test.shape}')
print(f'Columnas: {meta_test.columns.tolist()}')
print(f'\nNulos por columna:')
print(meta_test.isnull().sum())
print(f'\nn_titulaciones distribución:')
print(meta_test['n_titulaciones'].value_counts().sort_index())

meta_test shape: (6725, 12)
Columnas: ['titulacion', 'rama', 'sexo', 'pais_nombre', 'provincia', 'via_acceso', 'abandono', 'per_id_ficticio', 'curso_aca_ini', 'vive_fuera', 'cupo', 'n_titulaciones']

Nulos por columna:
titulacion           0
rama                 0
sexo                 0
pais_nombre          0
provincia            0
via_acceso           0
abandono             0
per_id_ficticio      0
curso_aca_ini        0
vive_fuera           0
cupo               709
n_titulaciones       0
dtype: int64

n_titulaciones distribución:
n_titulaciones
1    5672
2     979
3      68
4       3
5       3
Name: count, dtype: int64


In [6]:
# ============================================================================
# CELDA 6: FLAG DE CAUTELA — titulaciones con sesgo temporal
# ============================================================================
# Se incluyen TODAS las titulaciones. Las que tienen sesgo temporal se marcan
# con flag_cautela para que los módulos de Fase 6 puedan informar al lector.
#
# Criterios:
#   n_test < 30                → muestra insuficiente para métricas estables
#   Plan 2020 con 0% abandono  → alumnos sin tiempo suficiente de abandonar
#   Plan 2018 bajo abandono    → truncamiento temporal parcial
#   0% abandono sin plan nuevo → posible artefacto del dataset
# ============================================================================

TITULACIONES_CAUTELA = {
    'Doble Grado en Administración y Dirección de Empresas y Derecho,': 'n_test<30',
    'Grado en Criminologia y Seguridad  (Plan 2020)':                   'plan_reciente_0pct_abandono',
    'Grado en Arquitectura Técnica (Plan 2020)':                        'plan_reciente_0pct_abandono',
    'Grado en Ingeniería Agroalimentaria y del Medio Rural (Plan 2018)':'plan_reciente',
    'Grado en Maestro en Educación Primaria (Plan 2018)':               'plan_reciente_bajo_abandono',
    'Grado en Maestro en Educación Infantil (Plan 2018)':               'plan_reciente_bajo_abandono',
    'Grado en Ingeniería de la Edificación':                            '0pct_abandono_en_test',
}

meta_test['flag_cautela'] = meta_test['titulacion'].map(TITULACIONES_CAUTELA).fillna('ok')

print('Distribución flag_cautela:')
print(meta_test['flag_cautela'].value_counts())
print(f'\nObservaciones con cautela:   {(meta_test["flag_cautela"] != "ok").sum():,}')
print(f'Observaciones sin cautela:   {(meta_test["flag_cautela"] == "ok").sum():,}')

Distribución flag_cautela:
flag_cautela
ok                             6457
plan_reciente_bajo_abandono     186
0pct_abandono_en_test            41
plan_reciente_0pct_abandono      24
plan_reciente                    17
Name: count, dtype: int64

Observaciones con cautela:   268
Observaciones sin cautela:   6,457


In [7]:
# ============================================================================
# CELDA 7: VERIFICACIÓN FINAL
# ============================================================================
# Comprueba shape, índices y columnas esperadas antes de guardar.
# Si y_test.parquet existe, verifica que abandono coincide con el target real.
# ============================================================================

print('=' * 60)
print('VERIFICACIÓN FINAL DE meta_test')
print('=' * 60)

# Shape correcto
assert meta_test.shape[0] == len(X_test), \
    f'ERROR: {meta_test.shape[0]} filas en meta_test vs {len(X_test)} en X_test'
print(f'✅ Shape correcto: {meta_test.shape}')

# Índice alineado con X_test
assert list(meta_test.index) == list(X_test.index), \
    'ERROR: índices no coinciden con X_test'
print(f'✅ Índices alineados con X_test')

# Columnas esperadas
COLS_ESPERADAS = [
    'titulacion', 'rama', 'sexo', 'pais_nombre', 'provincia',
    'via_acceso', 'abandono', 'per_id_ficticio',
    'curso_aca_ini', 'vive_fuera', 'cupo',
    'n_titulaciones', 'flag_cautela'
]
print(f'\nColumnas:')
for col in COLS_ESPERADAS:
    print(f'  {"✅" if col in meta_test.columns else "❌"}  {col}')

# Verificar abandono vs y_test si existe
ruta_y = RUTA_MODEL / 'y_test.parquet'
if ruta_y.exists():
    y_test = pd.read_parquet(ruta_y)
    coincide = (meta_test['abandono'].values == y_test.values.ravel()).all()
    print(f'\n✅ abandono coincide con y_test: {coincide}')
else:
    print(f'\n⚠️  y_test.parquet no encontrado — abandono tomado de df_eda_final')
    print(f'   Distribución: {meta_test["abandono"].value_counts().to_dict()}')

print(f'\n✅ Verificación completada — listo para guardar')

VERIFICACIÓN FINAL DE meta_test
✅ Shape correcto: (6725, 13)
✅ Índices alineados con X_test

Columnas:
  ✅  titulacion
  ✅  rama
  ✅  sexo
  ✅  pais_nombre
  ✅  provincia
  ✅  via_acceso
  ✅  abandono
  ✅  per_id_ficticio
  ✅  curso_aca_ini
  ✅  vive_fuera
  ✅  cupo
  ✅  n_titulaciones
  ✅  flag_cautela

✅ abandono coincide con y_test: True

✅ Verificación completada — listo para guardar


In [8]:
# ============================================================================
# CELDA 8: GUARDAR meta_test.parquet
# ============================================================================
# Guardado en data/05_modelado/ junto con X_test_prep y y_test.
# Para cargar en cualquier módulo de Fase 6:
#   meta = pd.read_parquet(RUTA_MODEL / 'meta_test.parquet')
# ============================================================================


ruta_salida = RUTA_EVAL / 'meta_test.parquet'
meta_test.to_parquet(ruta_salida)

print(f'✅ Guardado: {ruta_salida}')
print(f'   Tamaño: {ruta_salida.stat().st_size / 1024:.1f} KB')

# Resumen ejecutivo
print(f'\n{"+" * 60}')
print(f'  RESUMEN meta_test.parquet')
print(f'{"+" * 60}')
print(f'  Observaciones en test:      {len(meta_test):,}')
print(f'  Alumnos únicos:             {meta_test["per_id_ficticio"].nunique():,}')
print(f'  Titulaciones:               {meta_test["titulacion"].nunique()} (todas incluidas)')
print(f'  Con 2+ titulaciones:        {(meta_test["n_titulaciones"] > 1).sum():,} '
      f'({(meta_test["n_titulaciones"] > 1).mean()*100:.1f}%)')
print(f'  Con flag cautela:           {(meta_test["flag_cautela"] != "ok").sum():,}')
print(f'  Tasa abandono en test:      {meta_test["abandono"].mean()*100:.1f}%')
print(f'  Cursos de ingreso:          '
      f'{sorted(meta_test["curso_aca_ini"].dropna().unique().astype(int).tolist())}')
print(f'  Viven fuera (%):            {meta_test["vive_fuera"].mean()*100:.1f}%')
print(f'{"+" * 60}')
print(f'\n🎯 meta_test.parquet listo para Fase 6.')

✅ Guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\06_evaluacion\meta_test.parquet
   Tamaño: 115.0 KB

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  RESUMEN meta_test.parquet
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Observaciones en test:      6,725
  Alumnos únicos:             6,603
  Titulaciones:               40 (todas incluidas)
  Con 2+ titulaciones:        1,053 (15.7%)
  Con flag cautela:           268
  Tasa abandono en test:      29.2%
  Cursos de ingreso:          [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
  Viven fuera (%):            12.9%
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

🎯 meta_test.parquet listo para Fase 6.


In [9]:
# ============================================================================
# CELDA 8b: GUARDAR X_test_prep_ids.parquet
# ============================================================================
# Fichero puente: indice posicional de Fase 5 + per_id_ficticio
# Permite join robusto en loaders.py usando per_id en lugar del indice.
# Estable aunque se re-ejecute Fase 5 con nuevos datos Excel.
# NO entra al modelo — solo sirve como puente de union.
# ============================================================================

x_test_ids = meta_test[["per_id_ficticio"]].copy()
x_test_ids.index = X_test.index

ruta_ids = RUTA_MODEL / "X_test_prep_ids.parquet"
x_test_ids.to_parquet(ruta_ids)

print("Guardado:", ruta_ids)
print("Shape:", x_test_ids.shape)
print("per_id_ficticio unicos:", x_test_ids["per_id_ficticio"].nunique())
print("Indice:", x_test_ids.index.min(), "-", x_test_ids.index.max())
print(x_test_ids.head())
print("X_test_prep_ids.parquet listo")


Guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\05_modelado\X_test_prep_ids.parquet
Shape: (6725, 1)
per_id_ficticio unicos: 6603
Indice: 16 - 33619
       per_id_ficticio
16823          1197964
4904            733138
11906           976958
25232          1506036
21637          1426114
X_test_prep_ids.parquet listo


In [10]:
# ============================================================================
# CELDA 9b: GUARDAR metricas_modelo.json — para la app Streamlit (Fase 7)
# ============================================================================
# Calcula métricas reales del modelo sobre X_test_prep (split de test de Fase 5)
# y estadísticas del dataset completo desde df_eda_final (ya cargado en celda 2).
# La app lee este JSON dinámicamente — sin valores hardcodeados.
# ============================================================================

import json as _json
import joblib
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score

# --- Cargar modelo y datos de test (split correcto de Fase 5) ---
ruta_modelo = RUTA_MODEL / 'models' / 'Stacking__balanced.pkl'
ruta_X_prep = RUTA_MODEL / 'X_test_prep.parquet'
ruta_y      = RUTA_MODEL / 'y_test.parquet'

assert ruta_modelo.exists(), f'❌ No encontrado: {ruta_modelo}'
assert ruta_X_prep.exists(), f'❌ No encontrado: {ruta_X_prep}'
assert ruta_y.exists(),      f'❌ No encontrado: {ruta_y}'

modelo           = joblib.load(ruta_modelo)
X_test_prep_eval = pd.read_parquet(ruta_X_prep)
y_test_eval      = pd.read_parquet(ruta_y).values.ravel()

# --- Predicciones sobre el split de test ---
prob_test = modelo.predict_proba(X_test_prep_eval)[:, 1]
pred_test = (prob_test >= 0.5).astype(int)

# --- Estadísticas del dataset COMPLETO (df_eda_final, ya cargado en celda 2) ---
# df_eda es el dataset D_strict completo: 33.621 registros
n_registros_total   = len(df_eda)
n_alumnos_total     = int(df_exp['per_id_ficticio'].nunique())  # alumnos únicos desde df_exp
tasa_abandono_total = float(df_eda['abandono'].mean()) if 'abandono' in df_eda.columns else float(y_test_eval.mean())
periodo_ini = 2010  # período del dataset D_strict (dato metodológico conocido)
periodo_fin = 2020  # datos completos hasta 2020

# === N_TEST CANÓNICO TRIBUNAL/MEMORIA (añadido 28/04/2026) ===
# La app necesita 2 cifras de tamaño de test:
#   - n_test_total = 6.725 → todas las filas del split de test
#   - n_test       = 6.596 → tras filtro implícito 2010-2020 (cifra canónica
#                            usada en memoria y defensa del TFM)
# Sin esto, p00_inicio.py contaba registros leyendo el parquet directamente
# y mostraba 6.725 (sin filtrar) en vez del 6.596 canónico.
n_test_total    = len(X_test_prep_eval)
n_test_canonico = int(meta_test['curso_aca_ini'].between(periodo_ini, periodo_fin).sum())

# --- Construir diccionario de métricas ---
metricas = {
    # Modelo
    'modelo_nombre':    'Stacking (CatBoost + RF + LogReg)',
    'auc':              round(float(roc_auc_score(y_test_eval, prob_test)), 4),
    'f1':               round(float(f1_score(y_test_eval, pred_test)), 4),
    'precision':        round(float(precision_score(y_test_eval, pred_test)), 4),
    'recall':           round(float(recall_score(y_test_eval, pred_test)), 4),
    'accuracy':         round(float(accuracy_score(y_test_eval, pred_test)), 4),
    # Dataset COMPLETO (D_strict)
    'n_registros':      n_registros_total,
    'n_alumnos_unicos': n_alumnos_total,
    'n_titulaciones':   int(meta_test['titulacion'].nunique()),
    # Tamaño del split de test (añadido 28/04/2026 — canónico tribunal)
    'n_test_total':     n_test_total,        # 6.725 sin filtrar
    'n_test':           n_test_canonico,     # 6.596 tras filtro 2010-2020
    'tasa_abandono':    round(tasa_abandono_total, 4),
    'periodo_inicio':   periodo_ini,
    'periodo_fin':      periodo_fin,
    # Baseline AutoML
    'baseline_nombre':  'CatBoost AutoML',
    'baseline_auc':     0.9270,
    'baseline_f1':      0.7970,
}

ruta_metricas = RUTA_EVAL / 'metricas_modelo.json'
with open(ruta_metricas, 'w', encoding='utf-8') as f:
    _json.dump(metricas, f, ensure_ascii=False, indent=2)

print(f'✅ Guardado: {ruta_metricas}')
print()
for k, v in metricas.items():
    print(f'  {k}: {v}')


✅ Guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\06_evaluacion\metricas_modelo.json

  modelo_nombre: Stacking (CatBoost + RF + LogReg)
  auc: 0.9535
  f1: 0.8273
  precision: 0.8483
  recall: 0.8073
  accuracy: 0.9014
  n_registros: 33621
  n_alumnos_unicos: 30872
  n_titulaciones: 40
  n_test_total: 6725
  n_test: 6596
  tasa_abandono: 0.2925
  periodo_inicio: 2010
  periodo_fin: 2020
  baseline_nombre: CatBoost AutoML
  baseline_auc: 0.927
  baseline_f1: 0.797
